In [2]:
from pathlib import Path
from PIL import Image, ImageFile
from tqdm import tqdm
import shutil

# If you want truncated images to be considered corrupt, leave this False.
# If you want to LOAD truncated images without error, set True.
ImageFile.LOAD_TRUNCATED_IMAGES = False

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}

def find_corrupt_images(root: str, move_bad_to: str | None = None):
    root = Path(root)
    bad = []

    # Collect paths first so tqdm can show total
    paths = [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ]

    for p in tqdm(paths, desc="Scanning images", unit="img"):
        try:
            # Open + verify header integrity
            with Image.open(p) as im:
                im.verify()

            # Re-open and force full decode (verify() doesn't decode pixel data)
            with Image.open(p) as im:
                im.load()

        except Exception as e:
            bad.append((str(p), repr(e)))

    print(f"Scanned {len(paths)} images. Found {len(bad)} corrupt.")

    if move_bad_to and bad:
        dst = Path(move_bad_to)
        dst.mkdir(parents=True, exist_ok=True)

        for path, _ in tqdm(bad, desc="Quarantining", unit="img"):
            src = Path(path)
            target = dst / src.name

            # avoid overwriting if same name
            i = 1
            while target.exists():
                target = dst / f"{src.stem}_{i}{src.suffix}"
                i += 1

            shutil.move(str(src), str(target))

    return bad

# Example:
# bad = find_corrupt_images("/path/to/images", move_bad_to="/path/to/quarantine")

bad = find_corrupt_images("../resources/cloud-images/NASA_GLOBE_CD/downloaded_images", move_bad_to="../resources/cloud-images/NASA_GLOBE_CD/downloaded_images/quarantined_images")
# Print a few examples
for p, err in bad[:20]:
    print(p, err)

Scanning images:  25%|██▍       | 117723/471443 [13:48<44:58, 131.08img/s] /Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Scanning images: 100%|██████████| 471443/471443 [56:28<00:00, 139.13img/s] 


Scanned 471443 images. Found 45 corrupt.


Quarantining: 100%|██████████| 45/45 [00:00<00:00, 2022.11img/s]


../resources/cloud-images/NASA_GLOBE_CD/downloaded_images/Sc/116-287952-94575966-202208251641_GW_Sc.jpg OSError('image file is truncated (2 bytes not processed)')
../resources/cloud-images/NASA_GLOBE_CD/downloaded_images/Sc/116-103008-31783502-202307110020_GE_Sc.jpg OSError('image file is truncated (1 bytes not processed)')
../resources/cloud-images/NASA_GLOBE_CD/downloaded_images/Sc/116-113101-3074308-202112062342_GN_Sc.jpg OSError('image file is truncated (8 bytes not processed)')
../resources/cloud-images/NASA_GLOBE_CD/downloaded_images/Sc/116-182796-55245232-201911271934_GE_Sc.jpg OSError('image file is truncated (74 bytes not processed)')
../resources/cloud-images/NASA_GLOBE_CD/downloaded_images/Sc/116-99589-31783502-202207270010_GU_Sc.jpg OSError('image file is truncated (6 bytes not processed)')
../resources/cloud-images/NASA_GLOBE_CD/downloaded_images/Sc/116-99589-31783502-202209062235_GU_Sc.jpg OSError('image file is truncated (2 bytes not processed)')
../resources/cloud-image